## First-time setup
Before running this script for the first time, check the Readme to install all required packages

In [ ]:
# Set to True to run only the final validation/export from existing intermediates to avoid rerunning all the aggregation steps.
RUN_ONLY_FINAL_VALIDATION = False
FINAL_VALIDATION_SKIP_MESSAGE = 'Final validation mode: intermediate files were created previously and aggregation is skipped.'

In [ ]:
# Environment preflight: requirements.txt is the single dependency list.
from importlib.util import find_spec
from pathlib import Path

requirements_path = Path('pipeline_support/requirements.txt')
if not requirements_path.is_file():
    raise RuntimeError(
        'Missing pipeline_support/requirements.txt. Run this notebook from the repository root.'
    )

# Package names do not always match their import names.
PACKAGE_IMPORT_NAMES = {
    'ipython': 'IPython',
    'ipykernel': 'ipykernel',
    'jupyter_client': 'jupyter_client',
    'nbclient': 'nbclient',
    'nbconvert': 'nbconvert',
    'nbformat': 'nbformat',
    'numpy': 'numpy',
    'openpyxl': 'openpyxl',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow',
}

missing_packages = []
for line in requirements_path.read_text(encoding='utf-8').splitlines():
    package = line.strip()
    if not package or package.startswith('#'):
        continue
    import_name = PACKAGE_IMPORT_NAMES.get(package.lower(), package)
    if find_spec(import_name) is None:
        missing_packages.append(package)

if missing_packages:
    missing_list = ', '.join(missing_packages)
    raise RuntimeError(
        'Missing required packages. From the repository root, run: '
        'pip install -r pipeline_support/requirements.txt. '
        f'Missing: {missing_list}'
    )

print('Environment preflight OK: pipeline_support/requirements.txt packages are installed.', flush=True)


In [ ]:
import json
import sys
import unicodedata
from contextlib import contextmanager
from pathlib import Path
from time import perf_counter
from uuid import uuid4

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display as ipy_display, Markdown, HTML

from pipeline_support.cgm_processing import create_cgm_parquet as build_cgm_parquet

RUN_ONLY_FINAL_VALIDATION = globals().get('RUN_ONLY_FINAL_VALIDATION', False)
FINAL_VALIDATION_SKIP_MESSAGE = globals().get(
    'FINAL_VALIDATION_SKIP_MESSAGE',
    'Final validation mode: aggregation cells will be skipped and existing intermediates will be validated/exported.',
)

class DataValidationError(Exception):
    """Raised when processed data fails an expected validation check."""


def install_validation_traceback_filter():
    # Keep validation failures fatal while hiding the noisy red traceback body.
    shell = get_ipython()
    if shell is None or getattr(shell, '_data_validation_traceback_filter_installed', False):
        return
    original_showtraceback = shell.showtraceback

    def filtered_showtraceback(
        exc_tuple=None,
        filename=None,
        tb_offset=None,
        exception_only=False,
        running_compiled_code=False,
    ):
        exception_type, exception_value, traceback = exc_tuple or sys.exc_info()
        if isinstance(exception_value, DataValidationError):
            return None
        return original_showtraceback(
            exc_tuple,
            filename,
            tb_offset,
            exception_only,
            running_compiled_code,
        )

    shell.showtraceback = filtered_showtraceback
    setattr(shell, '_data_validation_traceback_filter_installed', True)


install_validation_traceback_filter()


def fail_validation(message):
    # Show validation failures as notebook status messages before stopping execution.
    display(Markdown(
        f"### ❌ Data validation failed\n\n"
        f"⚠️ {message}\n\n"
        f"🛑 **Notebook execution stopped.**"
    ))
    raise DataValidationError(message)


# Formatting of tables
def display(obj):
    if isinstance(obj, pd.DataFrame):
        ipy_display(
            obj.style
            .hide(axis="index")
            .format(lambda x: f"{x:.3f}".rstrip("0").rstrip(".") if isinstance(x, float) else x)
        )
    else:
        ipy_display(obj)
# Same HTML style for all tables.
ipy_display(HTML("""
<style>
table {
    width: auto !important;
    table-layout: auto !important;
    border-left: 1px solid #666 !important;
    border-right: 1px solid #666 !important;
}

table th {
    white-space: nowrap !important;
    overflow: hidden !important;
    text-overflow: clip !important;
    text-align: center !important;
    border-top: 1px solid #777 !important;
    border-left: 1px solid #777 !important;
    border-right: 1px solid #777 !important;
    border-bottom: 1px solid #777 !important;
}

table td {
    white-space: nowrap !important;
    width: auto !important;
    text-align: center !important;
    border-left: 1px solid #666 !important;
    border-right: 1px solid #666 !important;
    border-bottom: 1px solid #666 !important;
}
</style>
"""))

# Resolve project root from current working directory.
cwd = Path.cwd()
PROJECT_ROOT = cwd

if (cwd / 'rawdata').is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'rawdata').is_dir():
    PROJECT_ROOT = cwd.parent
else:
    fail_validation(
        "Could not locate the project root because the required "
        "**rawdata** directory was not found.\n\n"
        f"Searched in:\n"
        f"- `{cwd / 'rawdata'}`\n"
        f"- `{cwd.parent / 'rawdata'}`"
    )


RAW_DIR = PROJECT_ROOT / 'rawdata'
IN_PROCESS_DIR = PROJECT_ROOT / 'inProcess'
FINAL_DATA_DIR = PROJECT_ROOT / 'finalData'

# Validate required input directories.
required_dirs = {
    'rawdata': RAW_DIR,
}

for name, directory_path in required_dirs.items():
    if not directory_path.is_dir():
        fail_validation(
            f"Required directory **{name}** was not found.\n\n"
            f"Expected location: `{directory_path}`"
        )

# Create output directories if needed.
IN_PROCESS_DIR.mkdir(exist_ok=True)
FINAL_DATA_DIR.mkdir(exist_ok=True)
INTERVAL = pd.Timedelta(minutes=5)
BATCH_SIZE = 500_000

# Fixed donation-level values known for every row in this dataset.
FIXED_DONATION_VALUES = {
    'cgm_device': 'Dexcom One',
    'insulin_delivery_algorithm': 'Basal-Bolus',
    'insulin_delivery_modality': 'SAP',
    'is_pregnant': False,
    'is_test': False,
    'subject_split_across_traintest': False,
    'source_file': 'SIESTA',
}

# Load standardized column names from Schema
SCHEMA_PATH = RAW_DIR / 'data_donation_schema.json'

if not SCHEMA_PATH.is_file():
    fail_validation(
        "Required schema file was not found: "
        f"`{SCHEMA_PATH}`"
    )

with SCHEMA_PATH.open(encoding='utf-8') as schema_file:
    SCHEMA = json.load(schema_file)

DONATION_COLUMNS = SCHEMA['donation_columns']
DONATION_SCHEMA = pa.schema([
    pa.field(column, pa.string() if column in SCHEMA['string_columns']
             else pa.bool_() if column in SCHEMA['boolean_columns'] else pa.float64())
    for column in DONATION_COLUMNS
])
# save valid ranges for variables
VALID_RANGES = {
    'CGM': (40, 400), 'age': (1, 120), 'basal': (0, 15), 'bolus': (0, 30),
    'galvanic_skin_response': (0, 100), 'heartrate': (30, 220), 'insulin': (0, 30),
    'skin_temp': (20, 42), 'weight': (20, 600), 'age_of_diagnosis': (0, 100),
}


@contextmanager
def staged_output(output_path):
    # Keeping the previous intermediate intact until its replacement is complete.
    temporary = output_path.with_name(f'.{output_path.stem}.{uuid4().hex}.partial.parquet')
    try:
        yield temporary
        temporary.replace(output_path)
    finally:
        # This also runs after an error or a keyboard interrupt.
        temporary.unlink(missing_ok=True)


def check_ranges(frame, columns, show_report=True):
    # Show missing values separately: missing data is not an out-of-range reading.
    report = pd.DataFrame([
        {
            'column': column,
            'missing': int(frame[column].isna().sum()),
            'out_of_range': int((frame[column].notna() & ~frame[column].between(*VALID_RANGES[column])).sum()),
            'min': frame[column].min(), 'max': frame[column].max(),
        }
        for column in columns
    ])
    # Successful checks can stay quiet when a combined summary follows later, using show_report=False.
    if show_report or report['out_of_range'].any():
        display(report)
    if report['out_of_range'].any():
        fail_validation('Out-of-range values remain; inspect the table above.')

## CGM backbone

The CGM conversion notebook owns the Excel import and its validation rules. We execute its code from the current notebook with a separate namespace, so its temporary variables do not leak here.

All subsequent joins use the same subject-specific five-minute grid. Outputs are intermediates under `inProcess`; final donation export and exercise integration remain pending.

In [ ]:
if RUN_ONLY_FINAL_VALIDATION:
    print(FINAL_VALIDATION_SKIP_MESSAGE)
else:
    def create_cgm_parquet():
        # Build the CGM backbone through the shared Python module, using this notebook's paths and validation helper.
        print('Creating CGM backbone...', flush=True)
        data_frame = build_cgm_parquet(
            project_root=PROJECT_ROOT,
            raw_dir=RAW_DIR,
            in_process_dir=IN_PROCESS_DIR,
            schema=SCHEMA,
            donation_columns=DONATION_COLUMNS,
            fail_validation=fail_validation,
            display=display,
        )
        with (IN_PROCESS_DIR / 'cgm_validation_summary.json').open(encoding='utf-8') as file:
            report = json.load(file)
        if not (
            report['status'] == 'CGM backbone OK'
            and report['conflicts_resolved'] and report['strict_5_minute_grid']
            and report['column_order_is_correct']
            and report['out_of_range_cgm_after_fixes'] == 0
        ):
            fail_validation(f'CGM validation failed: {report}')
        print(f"CGM ready: {report['total_rows']:,} rows, {report['participants']} subjects.", flush=True)
        return data_frame


    # Only the working table crosses from one processing section to the next.
    data = create_cgm_parquet()


## Join order and interval convention

`data` is the working table throughout this notebook. Demographics are joined by subject ID, followed by insulin and Empatica signals. Each row represents `(date - 5 minutes, date]`: an event at the right endpoint belongs to that row; an event at the left endpoint belongs to the previous row.

Section functions keep temporary variables local. The only additional result tables retained at notebook level are the Empatica audit and aggregation summaries. Raw files are read-only throughout.

## Demographic data integration

Demographic metadata is joined by subject `id`. The source file provides sex, age in March 2024, weight in kilograms, and year of type 1 diabetes diagnosis.

Mappings applied here:

- `age`: age in March 2024;
- `gender`: standardized `Male`/`Female`;
- `insulin_delivery_device`: pump model per subject, as confirmed by the data provider;
- `weight`: converted from kilograms to pounds;
- `age_of_diagnosis`: diagnosis year minus birth year.

In [ ]:
if RUN_ONLY_FINAL_VALIDATION:
    print(FINAL_VALIDATION_SKIP_MESSAGE)
else:
    def add_demographics(frame):
        # Match each subject once; convert source kilograms to the required pounds.
        with (RAW_DIR / 'DemographicData.json').open(encoding='utf-8') as file:
            records = json.load(file)

        # Pump models per subject, as confirmed by the data provider.
        pump_models = {
            '01': 'Paradigm 715',
            '02': 'Accu-Chek Guide Solo',
            '03': 'Paradigm 715',
            '04': 'Accu-Chek Guide Solo',
            '05': 'Accu-Chek Spirit Combo',
        }

        rows = []
        for record in records:
            # The nested birthday keys come from the source JSON, not a date convention
            # introduced here. Diagnosis age is approximate because only its year is known.
            birth_year = pd.to_datetime(
                record.get('Birthday (DD', {}).get('MM', {}).get('YYYY)'),
                dayfirst=True, errors='coerce',
            ).year
            rows.append({
                'id': str(record['Id']).zfill(2),
                'insulin_delivery_device': pump_models.get(str(record['Id']).zfill(2)),
                'age': pd.to_numeric(record.get('Age in March 2024'), errors='coerce'),
                'gender': {'M': 'Male', 'F': 'Female'}.get(record.get('Gender')),
                'weight': round(pd.to_numeric(record.get('Weight (kg)'), errors='coerce') * 2.2046226218, 3),
                'age_of_diagnosis': pd.to_numeric(record.get('Year of T1D diagnosis'), errors='coerce') - birth_year,
                **FIXED_DONATION_VALUES,
            })

        demographics = pd.DataFrame(rows)

        if demographics['insulin_delivery_device'].isna().any():
            fail_validation('Missing pump model for a subject; update pump_models.')

        if demographics['gender'].isna().any():
            fail_validation('Unknown gender value in DemographicData.json; expected M or F.')

        if not demographics['id'].is_unique or not frame['id'].isin(demographics['id']).all():
            fail_validation('Demographic IDs must be unique and cover every CGM subject.')

        demographics_by_id = demographics.set_index('id')
        result = frame.copy()

        for column in demographics_by_id:
            result[column] = result['id'].map(demographics_by_id[column])

        check_ranges(result, ['age', 'weight', 'age_of_diagnosis'], show_report=False)

        with staged_output(IN_PROCESS_DIR / 'data_with_demographics.parquet') as demographics_output:
            result.to_parquet(demographics_output, index=False)

        display(demographics)
        return result

    data = add_demographics(data)


## Exercise integration

Structured exercise sessions come from the CGM and Events workbook. Each event is recorded once on the CGM row that marks the event start interval: the nearest row for the same subject around the exercise `start_utc`. The full event duration is also recorded on that single row.

In [ ]:
if RUN_ONLY_FINAL_VALIDATION:
    print(FINAL_VALIDATION_SKIP_MESSAGE)
else:
    from openpyxl import load_workbook

    CGM_EVENTS_WORKBOOK_PATH = RAW_DIR / 'All patients CGM and Events.xlsx'


    def strip_accents(value):
        # Normalize workbook notes so matching is stable across accented and unaccented exports.
        return ''.join(
            character for character in unicodedata.normalize('NFKD', str(value))
            if not unicodedata.combining(character)
        )


    def classify_workout_label(notes):
        # The protocol notes identify only structured aerobic and resistance sessions.
        normalized_notes = strip_accents(notes).lower()
        if 'aerobico' in normalized_notes:
            return 'aerobic'
        if 'resistencia' in normalized_notes:
            return 'resistance'
        return pd.NA


    def iter_cgm_event_records(workbook_path):
        workbook = load_workbook(workbook_path, read_only=True, data_only=True)
        try:
            for subject_id in ['01', '02', '03', '04', '05']:
                sheet = workbook[subject_id]
                rows = sheet.iter_rows(values_only=True)
                # Workbook column names must be text. A formula object in a header
                # cannot identify an event field and should not be silently stringified.
                header_values = next(rows)
                if any(value is not None and not isinstance(value, str) for value in header_values):
                    fail_validation(f'Unexpected non-text event header in sheet {subject_id}.')
                headers = tuple(value.strip() if isinstance(value, str) else '' for value in header_values)
                for row_number, values in enumerate(rows, start=2):
                    record = dict(zip(headers, values, strict=False))
                    record['id'] = subject_id
                    record['row_number'] = row_number
                    yield record
        finally:
            workbook.close()


    def read_workout_events():
        # Read only structured treatment/exercise events from the workbook without modifying raw data.
        cgm_event_records = pd.DataFrame(iter_cgm_event_records(CGM_EVENTS_WORKBOOK_PATH))
        structured_exercise_events = cgm_event_records.loc[
            cgm_event_records['Origen'].astype(str).str.strip().str.lower().eq('treatments')
            & cgm_event_records['eventType'].astype(str).str.strip().str.lower().eq('exercise')
        ].copy()

        structured_exercise_events['notes'] = structured_exercise_events['notes'].fillna('').astype(str)
        structured_exercise_events['normalized_notes'] = structured_exercise_events['notes'].map(strip_accents).str.lower()
        structured_protocol_exercise = structured_exercise_events.loc[
            structured_exercise_events['normalized_notes'].str.contains('sesion estructurada', case=False, na=False)
        ].copy()
        structured_protocol_exercise['workout_label'] = structured_protocol_exercise['notes'].map(classify_workout_label)
        structured_protocol_exercise['start_utc'] = pd.to_datetime(
            structured_protocol_exercise['UNIX GTM+0'],
            unit='ms',
            errors='coerce',
            utc=True,
        )
        structured_protocol_exercise['duration_minutes'] = pd.to_numeric(
            structured_protocol_exercise['duration'],
            errors='coerce',
        )

        exercise_events = (
            structured_protocol_exercise
            .dropna(subset=['workout_label', 'start_utc', 'duration_minutes'])
            .rename(columns={'row_number': 'structured_row_number'})
            [['id', 'workout_label', 'start_utc', 'duration_minutes', 'structured_row_number']]
            .copy()
        )
        exercise_events['id'] = exercise_events['id'].astype('string').str.zfill(2)
        exercise_events['duration_minutes'] = exercise_events['duration_minutes'].round(3)
        exercise_events = exercise_events.sort_values(['id', 'start_utc', 'structured_row_number']).reset_index(drop=True)
        if exercise_events['duration_minutes'].le(0).any():
            fail_validation('Structured exercise events contain non-positive durations.')
        display(exercise_events.groupby(['id', 'workout_label'])['duration_minutes'].agg(['size', 'sum']).reset_index())
        return exercise_events


In [ ]:
if RUN_ONLY_FINAL_VALIDATION:
    print(FINAL_VALIDATION_SKIP_MESSAGE)
else:
    def add_exercise(frame, exercise_events):
        # Assign each exercise once to the nearest CGM row around its start timestamp.
        result = frame.copy()
        result['date_utc'] = pd.to_datetime(result['date'], errors='coerce', utc=True, format='mixed')
        if result['date_utc'].isna().any():
            fail_validation('The CGM grid contains unparsed timestamps before exercise integration.')

        result['workout_duration'] = 0.0
        result['workout_label'] = pd.NA
        assigned_records = []
        blockers = []

        for subject_id, subject_events in exercise_events.groupby('id'):
            subject_index = result.index[result['id'].eq(subject_id)]
            subject_rows = result.loc[subject_index].sort_values('date_utc')
            if subject_rows.empty:
                blockers.append(f'Exercise events found for unknown subject {subject_id}.')
                continue

            subject_dates = subject_rows['date_utc'].dt.floor('s').reset_index(drop=True)
            subject_original_index = subject_rows.index.to_series().reset_index(drop=True)
            for _, event_row in subject_events.iterrows():
                # Match at second precision and attach the event to the nearest CGM row.
                event_start = pd.Timestamp(event_row['start_utc']).floor('s')
                insertion_position = int(subject_dates.searchsorted(event_start, side='left'))
                candidate_positions = []
                if insertion_position > 0:
                    candidate_positions.append(insertion_position - 1)
                if insertion_position < len(subject_dates):
                    candidate_positions.append(insertion_position)
                if not candidate_positions:
                    blockers.append(
                        f"No CGM row near exercise start for subject {subject_id}, "
                        f"workbook row {event_row['structured_row_number']}."
                    )
                    continue

                event_start_datetime = event_start.to_pydatetime()
                candidate_distances = {
                    position: abs(
                        (
                            pd.Timestamp(subject_dates.iloc[position]).to_pydatetime()
                            - event_start_datetime
                        ).total_seconds()
                    )
                    for position in candidate_positions
                }
                def candidate_distance(position):
                    return candidate_distances[position]

                target_position = min(candidate_positions, key=candidate_distance)
                target_index = subject_original_index.iloc[target_position]
                assigned_records.append({
                    'id': subject_id,
                    'target_index': target_index,
                    'target_date': subject_dates.iloc[target_position],
                    'start_utc': event_row['start_utc'],
                    'workout_label': event_row['workout_label'],
                    'workout_duration': event_row['duration_minutes'],
                    'structured_row_number': event_row['structured_row_number'],
                })

        assignments = pd.DataFrame(assigned_records)
        if blockers:
            fail_validation('Exercise integration failed: ' + '; '.join(blockers))
        if not assignments.empty and assignments['target_index'].duplicated().any():
            display(assignments.loc[assignments['target_index'].duplicated(keep=False)].reset_index(drop=True))
            fail_validation('Multiple exercise events map to the same CGM row; review before combining labels or durations.')

        for _, assignment in assignments.iterrows():
            result.loc[assignment['target_index'], 'workout_duration'] = assignment['workout_duration']
            result.loc[assignment['target_index'], 'workout_label'] = assignment['workout_label']

        result = result.drop(columns=['date_utc'])
        result = result[DONATION_COLUMNS]
        if result.loc[result['workout_duration'].gt(0), 'workout_label'].isna().any():
            fail_validation('Workout duration was assigned without a workout label.')
        if result['workout_label'].notna().sum() != len(assignments):
            fail_validation('Workout labels must appear exactly once per structured exercise event.')
        if not assignments.empty:
            assigned_minutes = result.loc[result['workout_duration'].gt(0), 'workout_duration'].sum()
            expected_minutes = exercise_events['duration_minutes'].sum()
            if abs(assigned_minutes - expected_minutes) > 0.01:
                fail_validation('Assigned exercise duration does not match source event duration.')

        with staged_output(IN_PROCESS_DIR / 'data_with_demographics_exercise.parquet') as exercise_output:
            result.to_parquet(exercise_output, index=False)

        exercise_audit = assignments.copy()
        if not exercise_audit.empty:
            exercise_audit['start_to_cgm_seconds'] = (
                pd.to_datetime(exercise_audit['target_date'], utc=True).dt.floor('s')
                - pd.to_datetime(exercise_audit['start_utc'], utc=True).dt.floor('s')
            ).dt.total_seconds().round(3)
        display(Markdown('### Exercise integration checks'))
        display(pd.DataFrame([
            {
                'check': 'Structured events integrated',
                'result': len(assignments),
                'detail': 'Each event is assigned once to the nearest CGM row around its start timestamp.',
            },
            {
                'check': 'Rows with workout label',
                'result': int(result['workout_label'].notna().sum()),
                'detail': 'This must match the number of structured exercise events.',
            },
            {
                'check': 'Assigned minutes',
                'result': round(float(result['workout_duration'].sum()), 3),
                'detail': 'Full event durations are stored on the single start row.',
            },
        ]))
        display(exercise_audit[['id', 'structured_row_number', 'start_utc', 'target_date', 'start_to_cgm_seconds', 'workout_label', 'workout_duration']])
        return result


    data = add_exercise(data, read_workout_events())


## Insulin integration

Insulin data is joined subject by subject against the processed CGM backbone. Each CGM row represents the backward-looking interval `(date - 5 minutes, date]`.

The raw basal signal is a rate in U/hour. The exported `basal` column must be insulin units delivered inside each 5-minute interval, so each basal segment is converted as:

`basal_units = basal_rate_u_per_hour * overlap_hours_with_interval`

Bolus records are discrete delivered units, so `bolus` is the sum of bolus events inside `(date - 5 minutes, date]`. Total `insulin` is `basal + bolus` for the same subject and interval.

### Pump duplicate records

Some pump-export rows can repeat the same `id`, `kind`, `timestamp_utc`, and `value`. These repeated rows are kept visible in the preflight table because they reflect the pump download as received. Discrete bolus rows are not deduplicated: two bolus rows with the same timestamp contribute both delivered values to the backward-looking 5-minute interval. Basal rows are rate-change records rather than delivered-dose events, so exact duplicate basal rows are redundant for interval integration and are collapsed before calculating basal overlap durations.

In [ ]:
if RUN_ONLY_FINAL_VALIDATION:
    print(FINAL_VALIDATION_SKIP_MESSAGE)
else:
    def read_insulin():
        # Read pump events once and reuse the parsed timestamps during integration.
        insulin_path = RAW_DIR / 'insulin.parquet'
        events = pd.read_parquet(insulin_path)
        with pq.ParquetFile(insulin_path) as parquet:
            if len(events) != parquet.metadata.num_rows:
                fail_validation('Not all insulin rows were read.')
        for column in ['timestamp_utc', 'timestamp_local']:
            parsed = pd.to_datetime(events[column], errors='coerce', utc=True, format='mixed')
            print(f"Insulin {column}: {int(parsed.notna().sum()):,}/{len(events):,} parsed.")
            if parsed.isna().any():
                display(events.loc[parsed.isna()].head(20).reset_index(drop=True))
                fail_validation(f'Unparsed insulin timestamps in {column}.')
            if column == 'timestamp_utc':
                events['timestamp_utc_parsed'] = parsed
        events['value_numeric'] = pd.to_numeric(events['value'], errors='coerce')
        events['id'] = events['id'].astype('string').str.zfill(2)
        events['kind'] = events['kind'].astype('string').str.lower().str.strip()
        if events['value_numeric'].isna().any() or events['id'].isna().any():
            fail_validation('Insulin events contain missing IDs or non-numeric doses/rates.')
        if not events['kind'].isin(['basal', 'bolus']).all():
            fail_validation('Unknown insulin event kind.')
        display(events.groupby(['id', 'kind'])['value_numeric'].agg(['size', 'min', 'max']).reset_index())
        display(Markdown('### Repeated pump entries (boluses remain additive)'))
        display(events.loc[events.duplicated(
            ['id', 'kind', 'timestamp_utc_parsed', 'value_numeric'], keep=False
        ), ['id', 'kind', 'timestamp_utc', 'value', 'source_file', 'sheet', 'row_number']].head(30).reset_index(drop=True))
        return events


In [ ]:
if RUN_ONLY_FINAL_VALIDATION:
    print(FINAL_VALIDATION_SKIP_MESSAGE)
else:
    def add_insulin(frame, events):
        # Convert basal rates to interval units and sum all bolus doses.
        result = frame.copy()
        result['date_utc'] = pd.to_datetime(result['date'], errors='coerce', utc=True, format='mixed')
        result['interval_start_utc'] = result['date_utc'] - INTERVAL
        result['basal'] = pd.NA
        result['bolus'] = 0.0

        basal_duplicate_rows_collapsed = 0

        for subject_id, subject_index in result.groupby('id').groups.items():
            print(f'Integrating insulin for subject {subject_id}...', flush=True)
            subject_rows = result.loc[subject_index].sort_values('date_utc')
            subject_insulin = events.loc[events['id'] == subject_id].sort_values('timestamp_utc_parsed')

            # Basal entries describe a rate from this time onward. Repeating the same
            # rate change adds no insulin, whereas repeated bolus doses are both counted.
            basal_events_raw = subject_insulin.loc[subject_insulin['kind'] == 'basal'].copy()
            basal_events = basal_events_raw.drop_duplicates(
                subset=['timestamp_utc_parsed', 'value_numeric'],
                keep='first',
            ).copy()
            basal_duplicate_rows_collapsed += len(basal_events_raw) - len(basal_events)
            bolus_events = subject_insulin.loc[subject_insulin['kind'] == 'bolus'].copy()

            if not basal_events.empty:
                basal_events['next_timestamp_utc'] = basal_events['timestamp_utc_parsed'].shift(-1)
                basal_events['next_timestamp_utc'] = basal_events['next_timestamp_utc'].fillna(subject_rows['date_utc'].max())

                # Integrate only the part of each rate segment inside (start, end].
                # Preserve the existing rule: missing basal is unknown, not zero.
                basal_values = []
                for _, cgm_row in subject_rows.iterrows():
                    interval_start = cgm_row['interval_start_utc']
                    interval_end = cgm_row['date_utc']
                    active_basal = basal_events.loc[
                        (basal_events['timestamp_utc_parsed'] < interval_end)
                        & (basal_events['next_timestamp_utc'] > interval_start)
                    ]
                    if active_basal.empty:
                        basal_values.append(pd.NA)
                        continue

                    basal_units = 0.0
                    for _, basal_row in active_basal.iterrows():
                        overlap_start = max(pd.Timestamp(interval_start), pd.Timestamp(basal_row['timestamp_utc_parsed']))
                        overlap_end = min(pd.Timestamp(interval_end), pd.Timestamp(basal_row['next_timestamp_utc']))
                        overlap_hours = max((overlap_end - overlap_start).total_seconds(), 0) / 3600
                        basal_units += basal_row['value_numeric'] * overlap_hours
                    basal_values.append(basal_units)

                result.loc[subject_rows.index, 'basal'] = basal_values

            if not bolus_events.empty:
                bolus_values = []
                bolus_timestamps = bolus_events['timestamp_utc_parsed']
                for _, cgm_row in subject_rows.iterrows():
                    interval_start = cgm_row['interval_start_utc']
                    interval_end = cgm_row['date_utc']
                    interval_bolus = bolus_events.loc[
                        (bolus_timestamps > interval_start)
                        & (bolus_timestamps <= interval_end),
                        'value_numeric',
                    ].sum()
                    bolus_values.append(float(interval_bolus))
                result.loc[subject_rows.index, 'bolus'] = bolus_values

        result['basal'] = pd.to_numeric(result['basal'], errors='coerce')
        result['bolus'] = pd.to_numeric(result['bolus'], errors='coerce')
        result['insulin'] = result['basal'] + result['bolus']
        result.loc[result['basal'].isna() & result['bolus'].eq(0), 'insulin'] = pd.NA
        result.loc[result['basal'].isna() & result['bolus'].gt(0), 'insulin'] = result['bolus']
        result[['basal', 'bolus', 'insulin']] = result[['basal', 'bolus', 'insulin']].round(3)

        result = result.drop(columns=['date_utc', 'interval_start_utc'])
        result = result[DONATION_COLUMNS]
        check_ranges(result, ['basal', 'bolus', 'insulin'])
        with staged_output(IN_PROCESS_DIR / 'data_with_demographics_exercise_insulin.parquet') as insulin_output:
            result.to_parquet(insulin_output, index=False)
        print(f'Basal duplicate rows collapsed: {basal_duplicate_rows_collapsed}')
        print(f"Intervals with bolus: {result['bolus'].gt(0).sum():,}; missing insulin: {result['insulin'].isna().sum():,}")
        return result


    data = add_insulin(data, read_insulin())


## Empatica: one pass from raw samples to five-minute intervals

Each raw signal is read once, in blocks of 500,000 rows. Each block is cleaned, audited, written to the clean intermediate and immediately accumulated into five-minute sums and counts.

Progress includes rows written and aggregated, caps, outliers, invalid timestamps and elapsed time. Exact timestamp and row-count checks share the processing pass. Only the completed output footer is read to verify the written row count.

Both clean and five-minute files are staged while processing. Audit failures prevent their publication. Raw timestamps, raw values, source files and cleaning flags remain available in the clean intermediate.

In [ ]:
if RUN_ONLY_FINAL_VALIDATION:
    print(FINAL_VALIDATION_SKIP_MESSAGE)
else:
    # Margins express the agreed boundary policy in each signal's original units.
    # For example, temperature 43 is capped to 42; temperature 300 becomes NaN.

    EMPATICA_SIGNALS = {
        'heartrate': {'source': 'HR', 'margin': 5.0},
        'skin_temp': {'source': 'TEMP', 'margin': 2.0},
        'galvanic_skin_response': {'source': 'EDA', 'margin': 10.0},
    }


    def process_empatica(base_grid, signal, config):
        # Clean, audit, write and aggregate each raw block before reading the next.
        started = last_progress = perf_counter()
        lower, upper = VALID_RANGES[signal]
        result = base_grid[['id', 'date']].sort_values(['id', 'date']).reset_index(drop=True)
        dates = pd.to_datetime(result['date'], utc=True, format='mixed', errors='raise')
        # CGM preprocessing already validates this grid; only prepare it for sample assignment.
        grids = {
            subject: (rows.to_numpy(), dates.loc[rows].to_numpy(dtype='datetime64[ns]'))
            for subject, rows in result.groupby('id').groups.items()
        }
        counts = ['raw_samples', 'valid_samples', 'capped_low_samples', 'capped_high_samples', 'outlier_samples']
        for column in counts:
            result[column] = 0
        sums = np.zeros(len(result))
        sources = [set() for _ in range(len(result))]
        example_limit = 10
        examples = []
        example_count = 0
        report = dict.fromkeys([
            'read_rows', 'parsed_timestamp_rows', 'invalid_timestamp_rows',
            'fractional_timestamp_rows', 'non_null_values', 'invalid_id_rows',
            'raw_capped_low_samples', 'raw_capped_high_samples',
            'raw_outlier_samples_set_to_nan',
        ], 0)
        minimum, maximum = float('inf'), float('-inf')
        output = IN_PROCESS_DIR / f"empatica_{config['source'].lower()}_clean.parquet"
        # An explicit schema handles empty files and batches containing only nulls.
        schema = pa.schema([
            ('id', pa.string()), ('timestamp_utc', pa.timestamp('ns', tz='UTC')),
            ('raw_value', pa.float64()), ('source_file', pa.string()),
            ('timestamp_utc_raw', pa.string()), ('capped_low', pa.bool_()),
            ('capped_high', pa.bool_()), ('far_outlier', pa.bool_()),
            ('clean_value', pa.float64()),
        ])
        with pq.ParquetFile(RAW_DIR / f"{config['source']}.parquet") as raw:
            report['parquet_metadata_rows'] = raw.metadata.num_rows
            print(f"{signal}: cleaning, writing and aggregating {raw.metadata.num_rows:,} rows...", flush=True)
            with (
                staged_output(output) as temporary,
                staged_output(IN_PROCESS_DIR / f"empatica_{config['source'].lower()}_5min.parquet") as interval_output,
            ):
                with pq.ParquetWriter(temporary, schema, compression='snappy') as writer:
                    for batch in raw.iter_batches(
                        batch_size=BATCH_SIZE,
                        columns=['id', 'timestamp_utc', 'value_1', 'source_file'],
                    ):
                        frame = batch.to_pandas().rename(columns={'value_1': 'raw_value'})
                        frame['id'] = frame['id'].astype('string').str.zfill(2)
                        frame['timestamp_utc_raw'] = frame['timestamp_utc'].astype('string')
                        # A single inferred timestamp format previously lost fractional-second rows.
                        # Always request mixed parsing and treat any NaT as a failed audit.
                        frame['timestamp_utc'] = pd.to_datetime(
                            frame['timestamp_utc'], format='mixed', utc=True, errors='coerce'
                        )
                        frame['raw_value'] = pd.to_numeric(frame['raw_value'], errors='coerce').astype('float64')
                        frame['capped_low'] = frame['raw_value'].between(lower - config['margin'], lower, inclusive='left')
                        frame['capped_high'] = frame['raw_value'].between(upper, upper + config['margin'], inclusive='right')
                        frame['far_outlier'] = frame['raw_value'].notna() & ~frame['raw_value'].between(lower - config['margin'], upper + config['margin'])
                        frame['clean_value'] = frame['raw_value'].clip(lower, upper).mask(frame['far_outlier'])
                        report['read_rows'] += len(frame)
                        report['parsed_timestamp_rows'] += int(frame['timestamp_utc'].notna().sum())
                        report['invalid_timestamp_rows'] += int(frame['timestamp_utc'].isna().sum())
                        report['invalid_id_rows'] += int((frame['id'].isna() | frame['id'].str.strip().eq('')).sum())
                        report['fractional_timestamp_rows'] += int(
                            frame['timestamp_utc_raw'].str.contains('.', regex=False, na=False).sum()
                        )
                        report['non_null_values'] += int(frame['raw_value'].notna().sum())
                        for flag, counter in [
                            ('capped_low', 'raw_capped_low_samples'),
                            ('capped_high', 'raw_capped_high_samples'),
                            ('far_outlier', 'raw_outlier_samples_set_to_nan'),
                        ]:
                            report[counter] += int(frame[flag].sum())
                        if frame['raw_value'].notna().any():
                            minimum = min(minimum, frame['raw_value'].min())
                            maximum = max(maximum, frame['raw_value'].max())

                        # Keep only a few examples. The counters above include all rows,
                        # even after the example limit has been reached.
                        if example_count < example_limit:
                            sample = frame.loc[
                                frame[['capped_low', 'capped_high', 'far_outlier']].any(axis=1)
                                | frame['timestamp_utc'].isna()
                            ].head(example_limit - example_count).copy()
                            if not sample.empty:
                                examples.append(sample)
                                example_count += len(sample)
                        writer.write_table(pa.table(frame, schema=schema))
                        # Reuse this cleaned block immediately. No second Parquet scan
                        # is needed to compute the five-minute sums and sample counts.
                        # Invalid timestamps still block publication after the scan.
                        samples = frame.dropna(subset=['id', 'timestamp_utc'])
                        for subject, samples_for_subject in samples.groupby('id', sort=False):
                            if subject not in grids:
                                continue
                            rows, ends = grids[subject]
                            times = samples_for_subject['timestamp_utc'].to_numpy(dtype='datetime64[ns]')
                            positions = np.searchsorted(ends, times, side='left')
                            # Clipping is only for safe indexing. The first condition still
                            # excludes samples after the final interval of this subject.
                            inside = (positions < len(ends)) & (times > ends[positions.clip(max=len(ends) - 1)] - np.timedelta64(INTERVAL.value, 'ns'))
                            selected = samples_for_subject.loc[inside]
                            assigned = rows[positions[inside]]
                            values = selected['clean_value'].to_numpy(dtype=float)

                            # Add each sample to its CGM row. Counts keep missing observations
                            # separate from true zero values and make the mean auditable.
                            np.add.at(sums, assigned, np.where(np.isnan(values), 0.0, values))
                            for column, weights in zip(counts, [
                                np.ones(len(selected), dtype='int64'),
                                ~np.isnan(values), selected['capped_low'].to_numpy(),
                                selected['capped_high'].to_numpy(), selected['far_outlier'].to_numpy(),
                            ]):
                                result[column] += np.bincount(assigned, weights=weights, minlength=len(result)).astype('int64')

                            # Provenance is a small set per interval, not a list per sample.
                            for row, source in pd.DataFrame({
                                'row': assigned, 'source': selected['source_file'].to_numpy(),
                            }).dropna().drop_duplicates().itertuples(index=False, name=None):
                                sources[row].add(str(source))
                        if perf_counter() - last_progress >= 10: #report every 10 seconds
                            print(
                                f"  {report['read_rows']:,}/{raw.metadata.num_rows:,} rows written and aggregated; "
                                f"capped={report['raw_capped_low_samples'] + report['raw_capped_high_samples']:,}, "
                                f"outliers={report['raw_outlier_samples_set_to_nan']:,}, "
                                f"invalid timestamps={report['invalid_timestamp_rows']:,}; "
                                f"{perf_counter() - started:.1f}s",
                                flush=True,
                            )
                            last_progress = perf_counter()

                # Read the completed footer only; do not scan the newly written file again.
                with pq.ParquetFile(temporary) as written:
                    report['written_rows'] = written.metadata.num_rows
                report['all_rows_read'] = (
                    report['read_rows'] == report['written_rows'] == report['parquet_metadata_rows']
                )
                if not report['all_rows_read'] or report['invalid_timestamp_rows'] or report['invalid_id_rows']:
                    display(pd.DataFrame([report]))
                    if examples:
                        display(pd.concat(examples, ignore_index=True))
                    fail_validation(f'{signal}: audit failed; the previous intermediate was not replaced.')

                # Divide once after all blocks. Averaging block means would bias
                # intervals whose samples cross a block boundary.
                result[signal] = (sums / result['valid_samples'].replace(0, np.nan)).round(3)
                result['source_files'] = [';'.join(sorted(names)) if names else pd.NA for names in sources]
                result = result[['id', 'date', signal, *counts, 'source_files']]
                check_ranges(result, [signal], show_report=False)
                result.to_parquet(interval_output, index=False)

        total_example_rows = (
            report['raw_capped_low_samples']
            + report['raw_capped_high_samples']
            + report['raw_outlier_samples_set_to_nan']
            + report['invalid_timestamp_rows']
        )

        summary = {
            **report,
            'signal': signal,
            'raw_samples_assigned': int(result['raw_samples'].sum()),
            'samples_outside_cgm_grid': report['read_rows'] - int(result['raw_samples'].sum()),
            'valid_samples_used': int(result['valid_samples'].sum()),
            'assigned_capped_low_samples': int(result['capped_low_samples'].sum()),
            'assigned_capped_high_samples': int(result['capped_high_samples'].sum()),
            'assigned_outlier_samples': int(result['outlier_samples'].sum()),
            'non_null_5min_values': int(result[signal].notna().sum()),
            'example_rows_available': int(total_example_rows),
            'example_rows_displayed': int(example_count),
            'example_rows_not_displayed': max(0, int(total_example_rows) - int(example_count)),
            'min_raw_value': minimum if report['non_null_values'] else np.nan,
            'max_raw_value': maximum if report['non_null_values'] else np.nan,
            'elapsed_seconds': round(perf_counter() - started, 2),
            'output': str(output),
        }
        print(f"{signal}: clean and five-minute intermediates ready ({summary['elapsed_seconds']}s).", flush=True)
        if summary['example_rows_not_displayed']:
            print(
                f"{signal}: showing {summary['example_rows_displayed']} audit example rows; "
                f"{summary['example_rows_not_displayed']} additional rows are counted in the audit summary.",
                flush=True,
            )
        # The caller displays all signal summaries together after processing.
        if examples:
            display(pd.concat(examples, ignore_index=True))
        return result, summary


## Run Empatica processing

The following cell performs the single pass for HR, TEMP and EDA. Samples are assigned to `(date - 5 minutes, date]` for their subject. Valid sums and counts are accumulated across blocks; division happens once after the scan, so block size does not change the weighting.

The clean and five-minute intermediates are produced together under `inProcess`. Only the small five-minute tables are joined to `data`.

In [ ]:
if RUN_ONLY_FINAL_VALIDATION:
    print(FINAL_VALIDATION_SKIP_MESSAGE)
else:
    # This is where processing starts: one raw-data pass for each signal.
    # The function writes both intermediates and returns only the small interval table.
    empatica_reports = []
    for empatica_signal, empatica_config in EMPATICA_SIGNALS.items():
        empatica_values, empatica_report = process_empatica(data, empatica_signal, empatica_config)
        data = data.drop(columns=[empatica_signal]).merge(
            empatica_values[['id', 'date', empatica_signal]],
            on=['id', 'date'], how='left', validate='one_to_one', sort=False,
        )
        empatica_reports.append(empatica_report)

    # Keep the donation column order and save the combined intermediate.
    data = data[DONATION_COLUMNS]
    with staged_output(IN_PROCESS_DIR / 'data_with_demographics_exercise_insulin_empatica.parquet') as empatica_path:
        data.to_parquet(empatica_path, index=False)
    empatica_audit = pd.DataFrame(empatica_reports)
    display(empatica_audit)
    display(data.groupby('id')[list(EMPATICA_SIGNALS)].count().reset_index())


In [ ]:
SCHEMA_INTERMEDIATE_PATH = IN_PROCESS_DIR / 'data_with_demographics_exercise_insulin_empatica_schema.parquet'


def summarize_and_audit_intermediate(normalized, row_count, output_path):
    final_summary_rows = []
    for column in normalized.columns:
        observed_values = normalized[column].dropna().drop_duplicates().head(6).tolist()
        final_summary_rows.append({
            'column': column,
            'dtype': str(normalized[column].dtype),
            'missing_values': int(normalized[column].isna().sum()),
            'observed_values': observed_values if len(observed_values) <= 5 else 'more than 5 unique values',
        })
    final_summary = pd.DataFrame(final_summary_rows)
    display(final_summary)

    # Persist the same column-level audit shown above so the notebook summary
    # also leaves a file-based trace for review outside Jupyter.
    audit_csv_path = IN_PROCESS_DIR / 'final_validation_audit.csv'
    final_summary.to_csv(audit_csv_path, index=False)
    print(f'Final validation audit exported: {audit_csv_path.relative_to(PROJECT_ROOT)}')
    print(f'Validated intermediate: {row_count:,} rows, {len(normalized.columns)} columns -> {output_path.relative_to(PROJECT_ROOT)}')


def validate_existing_schema_intermediate(intermediate_output=SCHEMA_INTERMEDIATE_PATH):
    # Allows running the final validation/export cells after reopening the notebook,
    # as long as the schema-normalized intermediate parquet already exists on disk.
    if not intermediate_output.is_file():
        fail_validation(
            'The schema-validated intermediate does not exist yet. Run the full pipeline first.'
        )
    table = pq.read_table(intermediate_output)
    if not table.schema.equals(DONATION_SCHEMA):
        fail_validation('Existing schema-validated intermediate differs from the expected donation schema.')
    normalized = table.to_pandas()
    summarize_and_audit_intermediate(normalized, table.num_rows, intermediate_output)
    return normalized


def validate_and_write_intermediate(frame):
    # Check the complete working table before publishing its schema-normalized copy.
    # Donation subject IDs carry the study prefix; intermediates keep the bare numbers.
    frame = frame.copy()
    frame['id'] = 'SIESTA-' + frame['id'].astype('string').str.removeprefix('SIESTA-')
    if list(frame.columns) != DONATION_COLUMNS:
        fail_validation('Column order differs from the donation schema.')
    if frame['id'].isna().any() or frame.duplicated(['id', 'date']).any():
        fail_validation('Missing subject IDs or duplicate grid rows.')
    grid = frame[['id']].assign(date=pd.to_datetime(
        frame['date'], utc=True, format='mixed', errors='coerce'
    )).sort_values(['id', 'date'])
    if grid['date'].isna().any():
        fail_validation('The CGM grid contains unparsed timestamps.')
    if grid.groupby('id')['date'].diff().dropna().ne(INTERVAL).any():
        fail_validation('The subject grid is not strictly five minutes.')
    if not frame['is_test'].eq(False).fillna(False).all():
        fail_validation('All rows must remain marked as non-test data.')
    check_ranges(frame, VALID_RANGES)

    # Explicit types also cover entirely empty columns; Arrow must not infer
    # their types from whichever values happen to be present in this dataset.
    normalized = frame.copy()
    for field in DONATION_SCHEMA:
        if pa.types.is_string(field.type):
            normalized[field.name] = normalized[field.name].astype('string')
        elif pa.types.is_boolean(field.type):
            boolean_values = normalized[field.name]
            if pd.api.types.is_string_dtype(boolean_values) or boolean_values.dtype == object:
                boolean_values = boolean_values.astype('string').str.strip().str.lower().map({
                    'true': True,
                    'false': False,
                    '1': True,
                    '0': False,
                })
            normalized[field.name] = boolean_values.astype('boolean')
        else:
            normalized[field.name] = pd.to_numeric(normalized[field.name], errors='raise')
    normalized_table = pa.table(normalized, schema=DONATION_SCHEMA)

    with staged_output(SCHEMA_INTERMEDIATE_PATH) as intermediate_path:
        pq.write_table(normalized_table, intermediate_path)
        with pq.ParquetFile(intermediate_path) as parquet:
            if parquet.metadata.num_rows != len(frame) or not parquet.schema_arrow.equals(DONATION_SCHEMA):
                fail_validation('Written rows or Arrow schema differ from the expected output.')
    summarize_and_audit_intermediate(normalized, len(frame), SCHEMA_INTERMEDIATE_PATH)
    return normalized


if RUN_ONLY_FINAL_VALIDATION:
    data = validate_existing_schema_intermediate()
elif 'data' in globals():
    data = validate_and_write_intermediate(data)
else:
    data = validate_existing_schema_intermediate()


## Final donation export

This step publishes the schema-validated intermediate Parquet to `finalData` only after all checks above have passed.

In [ ]:
def publish_final_parquet():
    # Publish only the already schema-validated intermediate; do not rebuild or mutate prior stages here.
    intermediate_output = SCHEMA_INTERMEDIATE_PATH
    final_export_path = FINAL_DATA_DIR / 'SIESTA.parquet'
    table = pq.read_table(intermediate_output)
    if not table.schema.equals(DONATION_SCHEMA):
        fail_validation('Validated intermediate schema differs from the expected donation schema.')

    with staged_output(final_export_path) as final_path:
        pq.write_table(table, final_path)
        with pq.ParquetFile(final_path) as parquet:
            if parquet.metadata.num_rows != table.num_rows or not parquet.schema_arrow.equals(DONATION_SCHEMA):
                fail_validation('Final rows or Arrow schema differ from the validated intermediate.')

    print(f'Final donation export: {table.num_rows:,} rows, {len(table.column_names)} columns -> {final_export_path.relative_to(PROJECT_ROOT)}')
    return final_export_path


published_final_path = publish_final_parquet()